In [47]:
import sqlite3
import pandas as pd

# CD
conn = sqlite3.connect("banking.db")

# L
customers = pd.read_csv("/content/customers.csv")
accounts = pd.read_csv("/content/accounts.csv")
transactions = pd.read_csv("/content/transactions.csv")
loans = pd.read_csv("/content/loans.csv")

# L
customers.to_sql("customers", conn, if_exists="replace", index=False)
accounts.to_sql("accounts", conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
loans.to_sql("loans", conn, if_exists="replace", index=False)

print("Data loaded into SQLite successfully!")

Data loaded into SQLite successfully!


In [48]:

conn = sqlite3.connect("banking.db")

df = pd.read_sql("SELECT * FROM accounts LIMIT 5", conn)
print(df)

  account_id customer_id account_type    balance  status opened_date branch_id
0     A00000      C09416      Savings  490349.79  Active  03-06-2024      B030
1     A00001      C02406      Current  247306.73  Active  11-11-2025      B029
2     A00002      C09854      Current  473425.67  Active  08-09-2023      B041
3     A00003      C08555      Current  123445.12  Closed  22-01-2026      B003
4     A00004      C01076      Savings  403457.93  Closed  09-06-2024      B024


In [49]:
import sqlite3
import pandas as pd
from datetime import datetime
import os


conn = sqlite3.connect("banking.db")


queries = {

    # DQ
    "negative_balance": """
        SELECT * FROM accounts WHERE balance < 0
    """,

    "invalid_customer": """
        SELECT a.*
        FROM accounts a
        LEFT JOIN customers c
        ON a.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,

    "negative_transactions": """
        SELECT * FROM transactions WHERE amount < 0
    """,

    "invalid_account": """
        SELECT t.*
        FROM transactions t
        LEFT JOIN accounts a
        ON t.account_id = a.account_id
        WHERE a.account_id IS NULL
    """,

    "invalid_email": """
        SELECT * FROM customers
        WHERE email NOT LIKE '%@%.%'
    """,

    "underage_customers": """
        SELECT *
        FROM customers
        WHERE (strftime('%Y','now') - strftime('%Y', dob)) < 18
    """,

    #
    "duplicate_pan": """
        SELECT pan_number, COUNT(*) as count
        FROM customers
        GROUP BY pan_number
        HAVING COUNT(*) > 1
    """,

    "inactive_account_txn": """
        SELECT t.*
        FROM transactions t
        JOIN accounts a ON t.account_id = a.account_id
        WHERE a.status != 'active'
    """,

    "accounts_no_txn": """
        SELECT a.*
        FROM accounts a
        LEFT JOIN transactions t
        ON a.account_id = t.account_id
        WHERE t.account_id IS NULL
    """,

    # R/F
    "high_value_transactions": """
        SELECT * FROM transactions WHERE amount > 100000
    """,

    "multiple_txn_same_day": """
        SELECT account_id, txn_date, COUNT(*) as txn_count
        FROM transactions
        GROUP BY account_id, txn_date
        HAVING COUNT(*) > 5
    """,

    "loan_vs_balance_risk": """
        SELECT l.customer_id, l.loan_amount, a.balance
        FROM loans l
        JOIN accounts a ON l.customer_id = a.customer_id
        WHERE l.loan_amount > a.balance * 10
    """,

    # A
    "top_customers": """
        SELECT a.customer_id, SUM(t.amount) as total_txn
        FROM transactions t
        JOIN accounts a ON t.account_id = a.account_id
        GROUP BY a.customer_id
        ORDER BY total_txn DESC
        LIMIT 5
    """,

    "daily_txn_summary": """
        SELECT account_id, txn_date, SUM(amount) as daily_total
        FROM transactions
        GROUP BY account_id, txn_date
    """,

    "transaction_spike": """
        SELECT
            account_id,
            txn_date,
            amount,
            LAG(amount) OVER (PARTITION BY account_id ORDER BY txn_date) AS prev_amount
        FROM transactions
    """
}


def run_checks():
    results = {}

    for name, query in queries.items():
        try:
            df = pd.read_sql(query, conn)
            results[name] = df
        except Exception as e:
            print(f" Error in query {name}: {e}")

    return results



def calculate_scores():

    total_accounts = pd.read_sql("SELECT COUNT(*) as cnt FROM accounts", conn)['cnt'][0]
    total_txn = pd.read_sql("SELECT COUNT(*) as cnt FROM transactions", conn)['cnt'][0]

    neg_balance = len(pd.read_sql(queries["negative_balance"], conn))
    neg_txn = len(pd.read_sql(queries["negative_transactions"], conn))

    accounts_score = round((total_accounts - neg_balance) / total_accounts * 100, 2)
    txn_score = round((total_txn - neg_txn) / total_txn * 100, 2)

    return {
        "accounts_score": accounts_score,
        "transactions_score": txn_score,
        "run_date": datetime.now()
    }



def save_outputs(results, scores):

    if not os.path.exists("output"):
        os.makedirs("output")

    # Save each query result
    for name, df in results.items():
        df.to_csv(f"output/{name}.csv", index=False)

    # Save history
    history_file = "output/quality_history.csv"

    if os.path.exists(history_file):
        old = pd.read_csv(history_file)
        new = pd.concat([old, pd.DataFrame([scores])])
        new.to_csv(history_file, index=False)
    else:
        pd.DataFrame([scores]).to_csv(history_file, index=False)



def run_pipeline():
    print(" Running Advanced Data Quality Pipeline...")

    results = run_checks()
    scores = calculate_scores()
    save_outputs(results, scores)

    print(" Pipeline completed successfully!")



if __name__ == "__main__":
    run_pipeline()

 Running Advanced Data Quality Pipeline...
 Pipeline completed successfully!


In [50]:
import sqlite3

conn = sqlite3.connect("banking.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS data_lineage (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source TEXT,
    target TEXT,
    transformation TEXT,
    run_date TEXT
)
""")

conn.commit()

In [51]:
from datetime import datetime

lineage_data = [
    ("customers.csv", "customers", "CSV Load", datetime.now()),
    ("accounts.csv", "accounts", "CSV Load", datetime.now()),
    ("transactions.csv", "transactions", "CSV Load", datetime.now()),
    ("transactions", "high_value_transactions.csv", "SQL Filter: amount > 100000", datetime.now()),
    ("accounts", "negative_balance.csv", "SQL Filter: balance < 0", datetime.now())
]

cursor.executemany("""
INSERT INTO data_lineage (source, target, transformation, run_date)
VALUES (?, ?, ?, ?)
""", lineage_data)

conn.commit()

In [52]:
def log_lineage():

    from datetime import datetime

    lineage_entries = []

    for name in queries.keys():
        lineage_entries.append((
            "database_tables",
            f"{name}.csv",
            "SQL Data Quality Check",
            datetime.now()
        ))

    cursor = conn.cursor()

    cursor.executemany("""
    INSERT INTO data_lineage (source, target, transformation, run_date)
    VALUES (?, ?, ?, ?)
    """, lineage_entries)

    conn.commit()

In [53]:
df = pd.read_sql("SELECT * FROM data_lineage", conn)
print(df)


   id            source                       target  \
0   1     customers.csv                    customers   
1   2      accounts.csv                     accounts   
2   3  transactions.csv                 transactions   
3   4      transactions  high_value_transactions.csv   
4   5          accounts         negative_balance.csv   
5   6     customers.csv                    customers   
6   7      accounts.csv                     accounts   
7   8  transactions.csv                 transactions   
8   9      transactions  high_value_transactions.csv   
9  10          accounts         negative_balance.csv   

                transformation                    run_date  
0                     CSV Load  2026-05-01 09:05:09.229430  
1                     CSV Load  2026-05-01 09:05:09.229432  
2                     CSV Load  2026-05-01 09:05:09.229433  
3  SQL Filter: amount > 100000  2026-05-01 09:05:09.229433  
4      SQL Filter: balance < 0  2026-05-01 09:05:09.229433  
5                